In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt


from numerical_approximation.chemotaxis import (
    ArcParams,
    ArcPhiParams,
    Endpoint,
    ExternalBC,
    InternalNode,
    arc_mass,
    plot_density_snapshots,
    step_network,
    validate_network,
)

# Network with more than two exits (NM2E)

Bretti & Natalini (2018), §4.2.3, Figs 20–24, Table 2. 15 nodes, 21 arcs: two sources
(N0, N1) and three sinks (N12, N13, N14). Arcs are 1-based in the paper, 0-based here.

| Arcs | Connections (paper arc: nodes) |
|---|---|
| Short, $L = 0.5$ | 1: 0–2, 2: 1–3, 3: 2–5, 4: 3–5, 8: 5–7, 13: 7–9, 14: 7–10, 15: 7–11, 19: 9–12, 20: 10–13, 21: 11–14 |
| Long, $L = 10$ | 5: 2–4, 6: 3–6, 7: 4–7, 9: 4–8, 10: 5–8, 11: 6–7, 12: 6–8, 16: 8–9, 17: 8–10, 18: 8–11 |

| Node (p)                | $\xi_{i,j}$ |
|-------------------------|-------------|
| 2, 3, 4, 6, 9, 10, 11   | 1/3         |
| 5                       | 1/4         |
| 7, 8                    | 1/6         |

$$
\partial_x \phi_1(0, t) = \partial_x \phi_2(0, t) = -1\;\qquad
\partial_x \phi_{19}(L_{19}, t) = \partial_x \phi_{20}(L_{20}, t) = \partial_x \phi_{21}(L_{21}, t) = 1
$$

Homogeneous Neumann for $u, v$ at every external node (no inflow), so the mass is conserved.
For all $i$, $\phi_i(x, 0) = 0$ and $u_i(x, 0)$ is uniform random in $[0.45, 0.55]$.

In [2]:
# Space step shared by every arc: M_i = 9 on the short arcs and 199 on the long
# ones, ~4140 time steps to T = 180 (the paper doesn't report its step sizes).
h_grid = 0.05
T = 180

n_edges = 21
# 1-based in the paper -> 0-based here. The short arcs form the minimum-length
# subnetwork joining both sources to all three sinks through the trunk N5 -> N7.
short_arcs = [i - 1 for i in [1, 2, 3, 4, 8, 13, 14, 15, 19, 20, 21]]
L = [0.5 if i in short_arcs else 10 for i in range(n_edges)]
D = [1 for _ in range(n_edges)]
a = [1 for _ in range(n_edges)]
b = [0.1 for _ in range(n_edges)]
lam = math.sqrt(0.33)
chi = 1
_kappa = 1.0 # All the kappas are the same

# The scheme assumes the CFL condition with equality, h_i = 2*k*lam, on every
# arc. With one lam that means one h for all arcs, so every L_i must be a
# whole multiple of h_grid, and each arc gets its own M_i = L_i/h_grid - 1.
M = [round(L[i] / h_grid) - 1 for i in range(n_edges)]
assert all(math.isclose((M[i] + 1) * h_grid, L[i]) for i in range(n_edges)), "every L_i must be a multiple of h_grid"
h = [h_grid for _ in range(n_edges)]
k = h_grid / (2 * lam)
n_time_steps = round(T / k) + 1

arc_params = {i: ArcParams(lam=lam, h=h[i], k=k, chi=chi) for i in range(n_edges)}
phi_arc_params = {i: ArcPhiParams(D=D[i], a=a[i], b=b[i], h=h[i], k=k, M=M[i]) for i in range(n_edges)}

u_minus = {i: np.zeros((n_time_steps, M[i] + 2)) for i in range(n_edges)}
u_plus = {i: np.zeros((n_time_steps, M[i] + 2)) for i in range(n_edges)}
phi = {i: np.zeros((n_time_steps, M[i] + 2)) for i in range(n_edges)}

distribution_range = (0.45, 0.55)
rng = np.random.default_rng()
for i in range(n_edges):
    u0 = rng.uniform(distribution_range[0], distribution_range[1], size=M[i] + 2)
    u_minus[i][0, :] = u0 / 2
    u_plus[i][0, :] = u0 / 2

Set up the boundary conditions

In [3]:
# Each arc as (node at its j=0 end, node at its j=M+1 end), in paper order.
# Sources N0, N1 sit at the j=0 end of their arcs and sinks N12..N14 at the
# j=M+1 end, so the flux signs below match Eq. (50). N2..N11 are internal.
arc_nodes = [
    (0, 2), (1, 3), (2, 5), (3, 5),    # arcs 1-4
    (2, 4), (3, 6), (4, 7), (5, 7),    # arcs 5-8
    (4, 8), (5, 8), (6, 7), (6, 8),    # arcs 9-12
    (7, 9), (7, 10), (7, 11),          # arcs 13-15
    (8, 9), (8, 10), (8, 11),          # arcs 16-18
    (9, 12), (10, 13), (11, 14),       # arcs 19-21
]
assert len(arc_nodes) == n_edges
sources = [0, 1]
sinks = [12, 13, 14]

# Endpoints meeting at each internal node N2..N11.
# "left" is j=0, "right" is j=M+1
node_endpoints = {f"N{p}": [] for p in range(2, 12)}
for arc, (tail, head) in enumerate(arc_nodes):
    if f"N{tail}" in node_endpoints:
        node_endpoints[f"N{tail}"].append(Endpoint(arc, "left"))
    if f"N{head}" in node_endpoints:
        node_endpoints[f"N{head}"].append(Endpoint(arc, "right"))

# kappa (phi coupling) and xi (u/v coupling) are scoped to each node and
# keyed by the arcs meeting there; xi also needs the (i, i) self-pairs.
# xi = 1 / (number of arcs at the node) reproduces Table 2.
internal_nodes = []
for endpoints in node_endpoints.values():
    arcs = [ep.arc for ep in endpoints]
    internal_nodes.append(InternalNode(
        endpoints=endpoints,
        kappa={(i, j): _kappa for i in arcs for j in arcs if i != j},
        xi={(i, j): 1 / len(arcs) for i in arcs for j in arcs},
    ))

# External ends: food at both sources and all three sinks (Eq. 50)
source_ends = [Endpoint(arc, "left") for arc, (tail, _) in enumerate(arc_nodes) if tail in sources]
sink_ends = [Endpoint(arc, "right") for arc, (_, head) in enumerate(arc_nodes) if head in sinks]
external_bcs = (
    [ExternalBC(ep, kind="flux", phibar=-1.0) for ep in source_ends]   # phi_x(0,t) = -1 on arcs 1, 2
    + [ExternalBC(ep, kind="flux", phibar=1.0) for ep in sink_ends]    # phi_x(L,t) = 1 on arcs 19-21
)

# u/v boundary kind at the external ends only: no inflow anywhere.
bc_type = {tuple(ep): "neumann" for ep in source_ends + sink_ends}

validate_network(arc_params, bc_type, external_bcs, internal_nodes)

Solve the system

In [ ]:
record_every = 5
t_history = []
mass_history = {i: [] for i in range(n_edges)}

for n in range(n_time_steps - 1):
    if n % record_every == 0:
        t_history.append(n * k)
        for i in range(n_edges):
            mass_history[i].append(arc_mass(u_minus[i][n], u_plus[i][n], h[i]))

    step_network(
        u_minus, u_plus, phi, n,
        arc_params, phi_arc_params, bc_type,
        external_bcs, internal_nodes,
    )

## Density on the graph itself

In [ ]:
# Positions keyed by internal-node index (N2..N11 -> 0..9) or external end (arc, side).
# Sources on the left, sinks on the right; N8 (long arcs only) is pushed below
# the trunk N5 -> N7 so the short-arc subnetwork stays readable.
pos = {
    2: (-2.0, 1.0), 3: (-2.0, -1.0),
    4: (-1.0, 2.5), 5: (-1.0, 0.0), 6: (-1.0, -2.5),
    7: (0.5, 0.0), 8: (0.5, -4.0),
    9: (2.0, 1.0), 10: (2.0, 0.0), 11: (2.0, -1.0),
}
node_pos = {p: pos[int(name[1:])] for p, name in enumerate(node_endpoints)} | {
    (0, "left"): (-3.0, 1.0),    # N0, source
    (1, "left"): (-3.0, -1.0),   # N1, source
    (18, "right"): (3.0, 1.0),   # N12, sink
    (19, "right"): (3.0, 0.0),   # N13, sink
    (20, "right"): (3.0, -1.0),  # N14, sink
}
node_labels = {p: name for p, name in enumerate(node_endpoints)} | {
    (0, "left"): "N0", (1, "left"): "N1",
    (18, "right"): "N12", (19, "right"): "N13", (20, "right"): "N14",
}

snapshot_times = [10, 30, 60, 180]
snapshot_indices = [min(round(t / k), n_time_steps - 1) for t in snapshot_times]
# Skip any time whose row is no longer finite (a blown-up run)
finite_indices = [
    idx for idx in snapshot_indices
    if all(np.isfinite(u_minus[i][idx] + u_plus[i][idx]).all() for i in range(n_edges))
]
skipped = [round(idx * k, 2) for idx in snapshot_indices if idx not in finite_indices]
if skipped:
    print(f"not plotted (non-finite): t = {skipped}")

plot_density_snapshots(
    node_pos, internal_nodes, external_bcs, u_minus, u_plus, finite_indices, k,
    node_labels=node_labels,
    ncols=2,
    vmax=6,  # clip to the paper's colour scale; larger u saturates
)
plt.show()